In [1]:

#  Python code using `pandas`, `scikit-learn`, and `TensorFlow/Keras` to implement the pipeline step-by-step.
# ### Prerequisites

import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

In [2]:
# Supress Tensorflow warnings
# warnings.filterwarnings('ignore', category=FutureWarning)
# tf.get_logger().setLevel('ERROR')


# ---- Load Prep Data

path = "F:/Users/MSR/Documents/Datasets/twitter_training.csv"
# df0 = pd.read_csv(path, sep=";")
df0 = pd.read_csv(path)

# --- Read CSV File
print(f'Successfully loaded file: {path}\n')

# --- Display Column Type
print(" -- Column Info Initial -- ")
print(df0.head())

df0.info()



Successfully loaded file: F:/Users/MSR/Documents/Datasets/twitter_training.csv

 -- Column Info Initial -- 
   2401  Borderlands  Positive  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

  im getting on borderlands and i will murder you all ,  
0  I am coming to the borders and I will kill you...     
1  im getting on borderlands and i will kill you ...     
2  im coming on borderlands and i will murder you...     
3  im getting on borderlands 2 and i will murder ...     
4  im getting into borderlands and i can murder y...     
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74642 entries, 0 to 74641
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74642

In [5]:
### Step 1: Load `twitter_training.csv` and print 5 rows
# The DataFrame screenshot you shared has no headers (it starts immediately with index 0), so we need to assign column names manually. 
# Based on the data, the columns appear to be: `ID`, `Topic`, `Sentiment`, and `Text`.
df = df0.copy()

# Define column names since the CSV lacks a header
cols = ['ID', 'Topic', 'Sentiment', 'Text']

# Load the CSV
df = pd.read_csv(path , names=cols, header=None)

# Print 5 rows
print(df.head())

     ID        Topic Sentiment  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

                                                Text  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  


In [8]:
### Step 2: Print unique sentiment labels, remove 'Neutral' or 'Irrelevant'
# We need to see what the categories are and filter out the ones we don't want 
# (assuming we only want Positive and Negative for binary classification).

# Print unique labels and counts
print("Original Sentiments:\n", df['Sentiment'].value_counts())
print("_"*50)

# Filter out 'Neutral' and 'Irrelevant'
df = df[~df['Sentiment'].isin(['Neutral', 'Irrelevant'])]

print("Filtered Sentiments:\n", df['Sentiment'].value_counts())
print("_"*50)

Original Sentiments:
 Sentiment
Negative    22521
Positive    20814
Name: count, dtype: int64
__________________________________________________
Filtered Sentiments:
 Sentiment
Negative    22521
Positive    20814
Name: count, dtype: int64
__________________________________________________


In [9]:
### Step 3: Handle missing data, convert labels using `factorize()`
# 1.  **Drop NaNs:** Your snippet showed `Text` has fewer non-null counts (73956 vs 74642), so this step is crucial.
# 2.  **Factorize:** Convert 'Positive'/'Negative' strings into 0/1 integers.

# Drop missing values
df.dropna(inplace=True)

# Convert categorical labels to numeric codes
# This will turn 'Positive' and 'Negative' into 1 and 0 (or vice versa)
df['Sentiment_Code'] = pd.factorize(df['Sentiment'])[0]

print(df[['Sentiment', 'Sentiment_Code']].head())

  Sentiment  Sentiment_Code
0  Positive               0
1  Positive               0
2  Positive               0
3  Positive               0
4  Positive               0


In [10]:
### Step 4: Extract text column and fit the tokenizer
# The Tokenizer converts words into integers (indices) based on frequency.

# Extract text
texts = df['Text'].astype(str).values

# Initialize Tokenizer
# num_words limits the vocabulary to the most frequent words
tokenizer = Tokenizer(num_words=10000) 

# Fit tokenizer on the text
tokenizer.fit_on_texts(texts)

In [11]:
### Step 5: Convert text to sequences and pad them
# Neural networks require inputs of the same length.
# We convert text to lists of integers and then "pad" them (add zeros) to make them all the same length.

# Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences to ensure uniform length
max_len = 100 # You can choose a higher number based on data analysis
X = pad_sequences(sequences, maxlen=max_len)

# Define Target variable
y = df['Sentiment_Code'].values

In [12]:
### Step 6: Construct the Deep Learning Sequential Model (LSTM)
# An LSTM (Long Short-Term Memory) network is good for text because it remembers context from previous words.

model = Sequential()

# 1. Embedding Layer: Turns integer sequences into dense vectors
# input_dim = size of vocabulary + 1, output_dim = size of vector space
model.add(Embedding(input_dim=10000 + 1, output_dim=128, input_length=max_len))

# 2. LSTM Layer: The core recurrent layer
model.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))

# 3. Dense Layer: Output layer (1 unit for binary 0/1 classification)
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 128)          1280128   
                                                                 
 lstm (LSTM)                 (None, 64)                49408     
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 1,329,601
Trainable params: 1,329,601
Non-trainable params: 0
_________________________________________________________________


In [14]:
### Step 7: Train the model and evaluate
# We need to split data into training and testing sets before training.

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
history = model.fit(
    X_train, 
    y_train, 
    epochs=5, 
    batch_size=64, 
    validation_data=(X_test, y_test)
)

# Evaluate
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

Epoch 1/5
538/538 [==============================] - 357s 654ms/step - loss: 0.3659 - accuracy: 0.8338 - val_loss: 0.2287 - val_accuracy: 0.9063
Epoch 2/5
538/538 [==============================] - 333s 619ms/step - loss: 0.1707 - accuracy: 0.9286 - val_loss: 0.1846 - val_accuracy: 0.9244
Epoch 3/5
538/538 [==============================] - 727s 1s/step - loss: 0.1168 - accuracy: 0.9502 - val_loss: 0.1659 - val_accuracy: 0.9350
Epoch 4/5
538/538 [==============================] - 566s 1s/step - loss: 0.0955 - accuracy: 0.9576 - val_loss: 0.1675 - val_accuracy: 0.9318
Epoch 5/5
269/269 [==============================] - 15s 57ms/step - loss: 0.1667 - accuracy: 0.9396
Test Accuracy: 93.96%
